# Task 6: Music Recommendation System
**Slash Mark Machine Learning Internship - Advanced Level Project**

### Goal:
This notebook demonstrates how to design, implement, and evaluate a machine learning recommendation system for music tracks. It covers:
1. **Exploratory Data Analysis (EDA)** of Spotify audio features.
2. **Content-Based Filtering** (using Cosine Similarity on audio characteristics and genre).
3. **Collaborative Filtering** (using Singular Value Decomposition - SVD - on user-item interaction histories).
4. **Hybrid Recommendation System** (blending content similarity and collaborative taste profiling).

## 1. Setup & Data Loading

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import TruncatedSVD
from scipy.sparse import csr_matrix

# Set plots style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# File Paths
DATA_DIR = r"C:\Users\HP\Documents\KAIF\Slash\TASK 6\data"
SPOTIFY_DATA_PATH = os.path.join(DATA_DIR, "SpotifyFeatures.csv")
USER_DATA_PATH = os.path.join(DATA_DIR, "user_interactions.csv")

In [ ]:
# Load the datasets
df_tracks = pd.read_csv(SPOTIFY_DATA_PATH)
df_interactions = pd.read_csv(USER_DATA_PATH)

print(f"Tracks dataset: {df_tracks.shape[0]:,} songs, {df_tracks.shape[1]} columns")
print(f"User interaction logs: {df_interactions.shape[0]:,} entries")

Let's view the headers of the Spotify Features dataset:

In [ ]:
df_tracks.head(3)

And the headers of the user interaction logs:

In [ ]:
df_interactions.head(3)

## 2. Exploratory Data Analysis (EDA)
Let's look at the distribution of track popularity, genre counts, and correlation between various audio characteristics.

In [ ]:
# 1. Plot genre distribution
plt.figure(figsize=(14, 6))
sns.countplot(data=df_tracks, x='genre', order=df_tracks['genre'].value_counts().index, palette='viridis')
plt.xticks(rotation=45, ha='right')
plt.title('Distribution of Songs by Genre')
plt.xlabel('Genre')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# 2. Correlation heatmap of audio features
audio_features = ['popularity', 'acousticness', 'danceability', 'energy', 'instrumentalness', 'liveness', 'loudness', 'speechiness', 'tempo', 'valence']
plt.figure(figsize=(10, 8))
sns.heatmap(df_tracks[audio_features].corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap of Audio Features')
plt.tight_layout()
plt.show()

## 3. Content-Based Recommendation System
Content-based filtering recommends items similar to those preferred by the user in the past based on item attributes. Here, we build a feature representation using the normalized audio features and genre (weighted using dummy variables), and then compute the **Cosine Similarity** between a query track and all other tracks in the library.

### Mathematical Formula:
$$\text{Similarity}(A, B) = \cos(\theta) = \frac{A \cdot B}{\|A\| \|B\|} = \frac{\sum_{i=1}^n A_i B_i}{\sqrt{\sum_{i=1}^n A_i^2} \sqrt{\sum_{i=1}^n B_i^2}}$$

In [ ]:
# Normalize numerical audio features
scaler = MinMaxScaler()
scaled_features = scaler.fit_transform(df_tracks[audio_features])

# One-hot encode genres
genre_dummies = pd.get_dummies(df_tracks['genre'], prefix='genre').values * 1.5

# Combine features
feature_matrix = np.hstack([scaled_features, genre_dummies])
print(f"Feature matrix shape: {feature_matrix.shape}")

In [ ]:
def get_content_recommendations(track_name, artist_name=None, top_n=10):
    if artist_name:
        matches = df_tracks[(df_tracks['track_name'].str.lower() == track_name.lower()) & 
                            (df_tracks['artist_name'].str.lower() == artist_name.lower())]
    else:
        matches = df_tracks[df_tracks['track_name'].str.lower() == track_name.lower()]
        
    if matches.empty:
        print(f"Track '{track_name}' not found.")
        return pd.DataFrame()
        
    query_idx = matches['popularity'].idxmax()
    query_vector = feature_matrix[query_idx].reshape(1, -1)
    
    # Compute similarities on-the-fly
    similarities = cosine_similarity(query_vector, feature_matrix).flatten()
    
    # Get indices of top matches excluding query
    related_indices = np.argsort(similarities)[::-1]
    related_indices = [idx for idx in related_indices if idx != query_idx]
    
    recommendations = df_tracks.iloc[related_indices[:top_n]].copy()
    recommendations['similarity_score'] = similarities[related_indices[:top_n]]
    
    return recommendations[['track_name', 'artist_name', 'genre', 'popularity', 'similarity_score']]

# Test Content Recommendation
get_content_recommendations("Don't Let Me Be Lonely Tonight", "Joseph Williams", top_n=5)

## 4. Collaborative Filtering (Matrix Factorization - SVD)
Collaborative filtering builds recommendations by collecting preferences from many users. We apply **Singular Value Decomposition (SVD)** on the user-item interaction matrix, representing the user ratings as low-rank matrices of user and item latent factors.

### Mathematical Concept:
$$R \approx U \cdot \Sigma \cdot V^T$$
Where:
- $R$ is the $m \times n$ user-track interaction matrix (centered by subtracting user means).
- $U$ is an $m \times k$ orthogonal matrix representing user latent factors.
- $\Sigma$ is a $k \times k$ diagonal matrix of singular values.
- $V^T$ is a $k \times n$ orthogonal matrix representing track latent factors.

In [ ]:
# Prepare collaborative filtering mapping
unique_users = sorted(df_interactions['user_id'].unique())
unique_tracks = sorted(df_interactions['track_id'].unique())

user_to_idx = {uid: idx for idx, uid in enumerate(unique_users)}
track_to_idx = {tid: idx for idx, tid in enumerate(unique_tracks)}
idx_to_track = {idx: tid for tid, idx in track_to_idx.items()}

user_means = df_interactions.groupby('user_id')['rating'].mean().to_dict()
global_mean = df_interactions['rating'].mean()

# Build user-track centered sparse matrix
rows = df_interactions['user_id'].map(user_to_idx).values
cols = df_interactions['track_id'].map(track_to_idx).values
centered_ratings = [row['rating'] - user_means.get(row['user_id'], global_mean) for _, row in df_interactions.iterrows()]

interaction_matrix = csr_matrix((centered_ratings, (rows, cols)), shape=(len(unique_users), len(unique_tracks)))

# Apply Truncated SVD
svd = TruncatedSVD(n_components=25, random_state=42)
user_features = svd.fit_transform(interaction_matrix)
item_features = svd.components_.T

raw_predictions = np.dot(user_features, svd.components_)
prediction_std = np.std(raw_predictions)
print(f"Explained Variance by SVD: {svd.explained_variance_ratio_.sum()*100:.2f}%")

In [ ]:
def get_collaborative_recommendations(user_id, top_n=10):
    user_mean = user_means.get(user_id, global_mean)
    if user_id not in user_to_idx:
        # Cold start
        popular = df_tracks.sort_values(by='popularity', ascending=False).head(top_n)
        return popular[['track_name', 'artist_name', 'genre', 'popularity']]
        
    u_idx = user_to_idx[user_id]
    raw_residuals = np.dot(user_features[u_idx], svd.components_)
    predicted_ratings = user_mean + (raw_residuals / prediction_std) * 0.8
    predicted_ratings = np.clip(predicted_ratings, 1.0, 5.0)
    
    # Exclude already listened tracks
    listened_tracks = df_interactions[df_interactions['user_id'] == user_id]['track_id'].values
    listened_indices = [track_to_idx[tid] for tid in listened_tracks if tid in track_to_idx]
    predicted_ratings[listened_indices] = -1.0
    
    top_indices = np.argsort(predicted_ratings)[::-1][:top_n]
    rec_track_ids = [idx_to_track[idx] for idx in top_indices]
    
    recommendations = df_tracks[df_tracks['track_id'].isin(rec_track_ids)].copy()
    rating_map = {tid: predicted_ratings[track_to_idx[tid]] for tid in rec_track_ids}
    recommendations['predicted_rating'] = recommendations['track_id'].map(rating_map).round(1)
    
    return recommendations.sort_values(by='predicted_rating', ascending=False)[['track_name', 'artist_name', 'genre', 'popularity', 'predicted_rating']]

# Test Collaborative Filtering
get_collaborative_recommendations("USR_0001", top_n=5)

## 5. Hybrid Recommender
By combining the similarity scores of content filtering and predicted user ratings from collaborative filtering, we create a hybrid recommendation engine. This overcomes data sparsity and provides highly relevant recommendations based on both matching characteristics and other similar user profiles.

### Combined Scoring:
$$\text{Hybrid Score} = w \cdot \text{Similarity Score} + (1 - w) \cdot \text{Normalized Predicted Rating}$$

In [ ]:
def predict_single_rating(user_id, track_id):
    user_mean = user_means.get(user_id, global_mean)
    if user_id not in user_to_idx or track_id not in track_to_idx:
        return user_mean
    u_idx = user_to_idx[user_id]
    t_idx = track_to_idx[track_id]
    raw_pred = np.dot(user_features[u_idx], svd.components_[:, t_idx])
    predicted_rating = user_mean + (raw_pred / prediction_std) * 0.8
    return np.clip(predicted_rating, 1.0, 5.0)

def get_hybrid_recommendations(user_id, last_track_name, last_artist_name=None, top_n=10, content_weight=0.5):
    # 1. Content recommendations (wide list)
    content_recs = get_content_recommendations(last_track_name, last_artist_name, top_n=50)
    if content_recs.empty:
        return get_collaborative_recommendations(user_id, top_n=top_n)
        
    # Extract track metadata for content recommendations
    # Find matching original tracks to map back track_id
    merged_recs = content_recs.merge(df_tracks[['track_name', 'artist_name', 'track_id']], on=['track_name', 'artist_name'])
    
    hybrid_scores = []
    for _, row in merged_recs.iterrows():
        track_id = row['track_id']
        similarity = row['similarity_score']
        
        pred_rating = predict_single_rating(user_id, track_id)
        collab_score = (pred_rating - 1.0) / 4.0
        
        combined_score = (content_weight * similarity) + ((1.0 - content_weight) * collab_score)
        
        hybrid_scores.append({
            'track_name': row['track_name'],
            'artist_name': row['artist_name'],
            'genre': row['genre'],
            'popularity': row['popularity'],
            'similarity_score': round(similarity, 3),
            'predicted_rating': round(pred_rating, 1),
            'hybrid_score': round(combined_score, 3)
        })
        
    df_hybrid = pd.DataFrame(hybrid_scores)
    return df_hybrid.sort_values(by='hybrid_score', ascending=False).head(top_n)

# Test Hybrid System
get_hybrid_recommendations("USR_0001", "Don't Let Me Be Lonely Tonight", "Joseph Williams", top_n=5)